# CS-781 Phase 2 (L-NER) — Architecture 3: Transformer-XL, new BPE (a1_tokenization.py + vocab.json/merges.txt)

**What this notebook does**
1. Loads my Phase-1 Transformer-XL checkpoint (`txl_full/run/final.pt`) and my from-scratch BPE tokenizer.
2. Re-creates the exact Phase-1 text preprocessing (ASCII-fold + whitespace collapse) **while keeping a map back to the original characters**, so every BPE token knows its exact original `[start, end)`.
3. Checks that gold spans → BIO tags → spans comes back exactly (the *alignment upper bound*), and that the encoder in causal mode reproduces the Phase-1 model bit for bit.
4. Runs several experiments (one per adaptation method). Each one trains on fold1, picks the best epoch by **strict macro-F1 on fold2**, retrains on fold1+fold2 for that many epochs, predicts fold3, and writes `submission_<name>.zip`.
5. Writes `results.json` per experiment: val F1, per-label P/R/F1, error types (boundary/label/spurious), params, trainable params, peak GPU memory, train/inference time.


In [1]:
# ============================================================ 1. SETUP
import os, sys, json, time, shutil, subprocess, threading, pickle
from pathlib import Path

QUICK_TEST = False        # True: tiny end-to-end check (~10 min).  False: the real run (commit it).
PHASE1_HINT = "archit-3v2-newjson"   # substring of the attached Phase-1 notebook's input path

WORK = Path("/kaggle/working/p2"); WORK.mkdir(parents=True, exist_ok=True)
OUT  = WORK / ("runs_quick" if QUICK_TEST else "runs")

def find(name, must_contain=""):
    hits = sorted(p for p in Path("/kaggle/input").rglob(name) if PHASE1_HINT in str(p) and must_contain in str(p))
    return str(hits[0]) if hits else None

PHASE1_CKPT  = find("final.pt", "txl_full") or find("best.pt", "txl_full") or find("last.pt", "txl_full")
TXL_MODEL_PY = find("txl_model.py")
if find("a1_tokenization.py"):                     # new BPE: a1_tokenization.py + tok/vocab.json + tok/merges.txt
    TOK_PY = find("a1_tokenization.py"); v = find("vocab.json"); TOK_PATH = str(Path(v).parent) if v else None
else:                                              # old BPE: a1_tokenizer.py + tok/bpe_tokenizer.json
    TOK_PY = find("a1_tokenizer.py"); TOK_PATH = find("bpe_tokenizer.json")
print("checkpoint :", PHASE1_CKPT); print("model code :", TXL_MODEL_PY); print("tokenizer  :", TOK_PY, "|", TOK_PATH)
assert all([PHASE1_CKPT, TXL_MODEL_PY, TOK_PY, TOK_PATH]), f"attach the Phase-1 notebook whose input path contains '{PHASE1_HINT}' (Add Input -> Your Work -> Notebooks), or fix PHASE1_HINT"

from kaggle_secrets import UserSecretsClient
HF_TOKEN = None
for k in ["HF", "HF_TOKEN"]:
    try: HF_TOKEN = UserSecretsClient().get_secret(k); break
    except Exception: pass
assert HF_TOKEN, "add your Hugging Face token as a Kaggle secret named HF (Add-ons -> Secrets) and tick it for this notebook"

from huggingface_hub import snapshot_download
DATA = Path(snapshot_download("Exploration-Lab/CS-781-Capstone", repo_type="dataset", token=HF_TOKEN,
                              allow_patterns=["Phase_2_LNER/*"], local_dir="/kaggle/working/hf_data")) / "Phase_2_LNER"
print(sorted(str(p.relative_to(DATA)) for p in DATA.rglob("*.json")))
LABELS = [l.strip() for l in open(DATA / "Train" / "labels.txt") if l.strip()]
print("labels:", LABELS)


checkpoint : /kaggle/input/notebooks/abhishek786595/archit-3v2-newjson/txl_full/run/final.pt
model code : /kaggle/input/notebooks/abhishek786595/archit-3v2-newjson/txl/txl_model.py
tokenizer  : /kaggle/input/notebooks/abhishek786595/archit-3v2-newjson/txl/a1_tokenization.py | /kaggle/input/notebooks/abhishek786595/archit-3v2-newjson/txl/tok


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

['Test/fold3-text.json', 'Train/fold1-gold.json', 'Train/fold1-text.json', 'Train/fold2-gold.json', 'Train/fold2-text.json']
labels: ['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO']


In [2]:
%%writefile /kaggle/working/p2/p2_ner.py
"""CS-781 Phase 2 (L-NER) on top of the Phase-1 Transformer-XL (either of my from-scratch BPE tokenizers, 32k vocab).

One file, used two ways:
  * imported by the notebook  -> data prep, alignment checks, model-equivalence test
  * run as a script            -> `python p2_ner.py job.json` trains/evaluates/predicts ONE experiment

Pipeline
  text --(ASCII-fold + whitespace-collapse, exactly like Phase-1 pretraining, but keeping a char map)-->
  BPE tokens with ORIGINAL character offsets --> BIO tags (25) --> 512-token sliding windows -->
  Transformer-XL encoder (+ PEFT) + token head --> overlap-weighted log-prob merge per document -->
  BIO-constrained Viterbi --> character spans (whitespace-stripped) --> prediction.json

Adaptation methods (METHOD):
  full     full fine-tuning
  partial  top-N blocks + head (layer-wise / partial fine-tuning)
  lora     LoRA:  y = W x + s*B(A x)
  lorma    LoRMA_+ (Bihany, Patel & Modi 2025, multiplicative):  y = (I + s*B A) W x   (B = 0 -> identity at init)
  adapter  Houlsby-style bottleneck adapters after attention and FFN sub-layers
  bitfit   biases (+ LayerNorm and attention u/v biases) only
  kd       knowledge distillation: a fine-tuned 14-layer teacher -> a shallower student built from Phase-1 layers
"""
import json
import math
import os
import pickle
import re
import sys
import time
import unicodedata
import zipfile
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np

# ============================================================== text normalisation with char map
_MAP = str.maketrans({
    "‘": "'", "’": "'", "‚": "'", "‛": "'", "“": '"', "”": '"',
    "„": '"', "«": '"', "»": '"', "–": "-", "—": "-", "−": "-",
    "‐": "-", "‑": "-", "‒": "-", "―": "-", " ": " ", "•": "*",
    "·": "*", "…": "...", "₹": "Rs. ",
})
_CTRL = re.compile(r"[\x00-\x08\x0e-\x1f\x7f]")
# Phase-1 pre-tokenizer WITH the underscore fix that the Phase-1 notebook applied
PRE_RE = re.compile(r" ?[^\W\d_]+| ?\d+| ?(?:[^\s\w]|_)+|\s+")


def normalize_with_map(text):
    """Same output as Phase-1 `normalize_text` (ASCII fold, straight quotes, single spaces, no newlines),
    plus omap[i] = index in the ORIGINAL text of normalised char i."""
    out, omap = [], []
    pend = -1                                   # orig index of the first whitespace in a pending run
    for i, ch in enumerate(text):
        s = ch
        if not ch.isascii():
            s = ch.translate(_MAP)
            if not s.isascii():
                s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("ascii")
        if s and _CTRL.search(s):
            s = _CTRL.sub("", s)
        for c in s:
            if c.isspace():
                if pend < 0:
                    pend = i
            else:
                if pend >= 0 and out:
                    out.append(" ")
                    omap.append(pend)
                pend = -1
                out.append(c)
                omap.append(i)
    return "".join(out), omap


# ============================================================== tokenizer with offsets
class BPE:
    """Uniform wrapper around my two from-scratch Phase-1 tokenizers.
      kind='old' : a1_tokenizer.py  + bpe_tokenizer.json   (<pad>0 <unk>1 <s>2 </s>3; underscore-fixed regex)
      kind='new' : a1_tokenization.py + vocab.json/merges.txt (<pad>0 <s>1 </s>2 <unk>3; GPT-2 style regex)
    Exposes: pre_re, encode_piece(piece) -> ids, nbytes[id], vocab_size, encode_full(text) (the Phase-1 path)."""

    def __init__(self, tokenizer_py, tok_path):
        import importlib.util
        name = Path(tokenizer_py).stem
        spec = importlib.util.spec_from_file_location(name + "_p2", tokenizer_py)
        mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mod)
        self.mod = mod
        if hasattr(mod, "BPETokenizer"):                              # old tokenizer
            self.kind = "old"
            p = Path(tok_path)
            self.tok = mod.BPETokenizer.load(str(p if p.is_file() else p / "bpe_tokenizer.json"))
            self.pre_re = PRE_RE
            self.encode_piece = self.tok._encode_pretoken
            self.nbytes = [len(b) for b in self.tok.id_to_bytes]
            self.vocab_size = self.tok.get_vocab_size()
        else:                                                         # new tokenizer
            self.kind = "new"
            p = Path(tok_path)
            self.tok = mod.Tokenizer.load(str(p.parent if p.is_file() else p))
            self.pre_re = mod.PRETOKEN_RE
            self.encode_piece = self.tok._encode_word
            n = mod.N_SPECIAL
            self.nbytes = [0] * n + [len(t) for t in self.tok.id_to_token[n:]]   # 1 char = 1 byte (GPT-2 byte map)
            self.vocab_size = self.tok.vocab_size

    def encode_full(self, norm_text):
        """Exactly what Phase-1 pretraining did with an already ASCII-normalised text."""
        out = []
        for piece in self.pre_re.findall(norm_text):
            out.extend(self.encode_piece(piece))
        return out


def load_bpe(tokenizer_py, tok_path):
    return BPE(tokenizer_py, tok_path)


def encode_doc(tok, text):
    """-> dict(ids int32[n], cs int32[n], ce int32[n]).  For token t, the ORIGINAL-text span is [cs, ce),
    where cs skips the single leading space a BPE token may carry."""
    norm, omap = normalize_with_map(text)
    ids, cs, ce = [], [], []
    for m in tok.pre_re.finditer(norm):
        piece, p = m.group(), m.start()
        for tid in tok.encode_piece(piece):
            L = tok.nbytes[tid]
            a, b = p, p + L
            p = b
            a2 = a + 1 if (norm[a] == " " and b > a + 1) else a
            ids.append(tid)
            cs.append(omap[a2])
            ce.append(omap[b - 1] + 1)
        assert p == m.end(), "token byte lengths do not add up (non-ASCII after normalisation?)"
    return dict(ids=np.asarray(ids, np.int32), cs=np.asarray(cs, np.int32), ce=np.asarray(ce, np.int32))


# ============================================================== labels / BIO
def make_tags(labels):
    tags = ["O"] + [f"B-{l}" for l in labels] + [f"I-{l}" for l in labels]
    return tags, {t: i for i, t in enumerate(tags)}


def strip_span(text, s, e):
    while s < e and text[s].isspace():
        s += 1
    while e > s and text[e - 1].isspace():
        e -= 1
    return s, e


def bio_from_gold(text, enc, ents, tag2id):
    """Token BIO ids from character-level gold entities (entities are whitespace-stripped first)."""
    owner = np.full(len(text) + 1, -1, np.int32)
    lab_of = []
    for k, e in enumerate(ents):
        s, en = strip_span(text, int(e[0]), int(e[1]))
        owner[s:en] = k
        lab_of.append(e[2])
    cs = enc["cs"]
    y = np.zeros(len(cs), np.int64)
    prev = -1
    for t in range(len(cs)):
        k = owner[cs[t]] if cs[t] < len(text) and not text[cs[t]].isspace() else -1
        if k >= 0:
            y[t] = tag2id[("I-" if k == prev else "B-") + lab_of[k]]
        prev = k
    return y


def spans_from_tags(text, enc, tag_ids, tags):
    out, cur = [], None                     # cur = [first_tok, last_tok, label]
    for t, ti in enumerate(tag_ids):
        tg = tags[ti]
        if tg == "O":
            if cur: out.append(cur); cur = None
            continue
        bi, lab = tg.split("-", 1)
        if bi == "B" or cur is None or cur[2] != lab:
            if cur: out.append(cur)
            cur = [t, t, lab]
        else:
            cur[1] = t
    if cur: out.append(cur)
    res = []
    for a, b, lab in out:
        s, e = strip_span(text, int(enc["cs"][a]), int(enc["ce"][b]))
        if e > s:
            res.append([s, e, lab])
    return res


def bio_transitions(tags):
    """allowed[i, j] for tag i -> tag j; start_ok[j]."""
    T = len(tags)
    allowed = np.ones((T, T), bool)
    start_ok = np.ones(T, bool)
    for j, tj in enumerate(tags):
        if tj.startswith("I-"):
            lab = tj[2:]
            start_ok[j] = False
            for i, ti in enumerate(tags):
                allowed[i, j] = ti in (f"B-{lab}", f"I-{lab}")
    return allowed, start_ok


def viterbi(logp, allowed, start_ok):
    """logp [n, T] log-probs -> best BIO-valid tag sequence."""
    n, T = logp.shape
    if n == 0:
        return np.zeros(0, np.int64)
    trans = np.where(allowed, 0.0, -1e9)
    score = np.where(start_ok, logp[0], -1e9)
    back = np.zeros((n, T), np.int32)
    for t in range(1, n):
        cand = score[:, None] + trans            # [from, to]
        back[t] = cand.argmax(0)
        score = cand.max(0) + logp[t]
    best = np.zeros(n, np.int64)
    best[-1] = int(score.argmax())
    for t in range(n - 1, 0, -1):
        best[t - 1] = back[t, best[t]]
    return best


# ============================================================== metric (strict span + label)
def strict_scores(gold, pred, labels):
    tp, fp, fn = Counter(), Counter(), Counter()
    for did in gold:
        g = {(int(s), int(e), l) for s, e, l, *_ in gold[did]}
        p = {(int(s), int(e), l) for s, e, l, *_ in pred.get(did, [])}
        for x in g & p: tp[x[2]] += 1
        for x in p - g: fp[x[2]] += 1
        for x in g - p: fn[x[2]] += 1
    per = {}
    for l in labels:
        P = tp[l] / (tp[l] + fp[l]) if tp[l] + fp[l] else 0.0
        R = tp[l] / (tp[l] + fn[l]) if tp[l] + fn[l] else 0.0
        F = 2 * P * R / (P + R) if P + R else 0.0
        per[l] = dict(p=P, r=R, f1=F, tp=tp[l], fp=fp[l], fn=fn[l])
    mac = lambda k: sum(per[l][k] for l in labels) / len(labels)
    return dict(macro_f1=mac("f1"), macro_p=mac("p"), macro_r=mac("r"), per_label=per)


def error_breakdown(gold, pred):
    """Classify every false positive: label error (same span, other label), boundary error
    (overlaps a gold span of the same label), or spurious; plus the label-confusion counts."""
    kinds, conf = Counter(), Counter()
    for did in gold:
        g = [(int(s), int(e), l) for s, e, l, *_ in gold[did]]
        gs = set(g)
        for s, e, l in [(int(s), int(e), l) for s, e, l, *_ in pred.get(did, [])]:
            if (s, e, l) in gs:
                continue
            same = [gl for (a, b, gl) in g if a == s and b == e]
            if same:
                kinds["label_error"] += 1; conf[f"{same[0]}->{l}"] += 1
            elif any(a < e and s < b and gl == l for a, b, gl in g):
                kinds["boundary_error"] += 1
            elif any(a < e and s < b for a, b, gl in g):
                kinds["overlap_other_label"] += 1
            else:
                kinds["spurious"] += 1
    return dict(fp_kinds=dict(kinds), top_confusions=conf.most_common(15))


# ============================================================== data prep (called from the notebook)
def prepare(data_dir, tok, labels, cache_path):
    data_dir = Path(data_dir)
    tags, tag2id = make_tags(labels)
    out = {}
    for fold in ["fold1", "fold2", "fold3"]:
        sub = "Test" if fold == "fold3" else "Train"
        texts = json.load(open(data_dir / sub / f"{fold}-text.json", encoding="utf-8"))
        gold = json.load(open(data_dir / sub / f"{fold}-gold.json", encoding="utf-8")) if fold != "fold3" else None
        docs = {}
        for did, text in texts.items():
            enc = encode_doc(tok, text)
            d = dict(text=text, **enc)
            if gold is not None:
                d["gold"] = gold.get(did, [])
                d["y"] = bio_from_gold(text, enc, d["gold"], tag2id)
            docs[did] = d
        out[fold] = docs
    pickle.dump(dict(labels=labels, tags=tags, folds=out), open(cache_path, "wb"))
    return out


def alignment_report(folds, labels):
    """Upper bound of strict macro-F1 if the model predicted the gold BIO tags perfectly."""
    tags, _ = make_tags(labels)
    rep = {}
    for fold in ["fold1", "fold2"]:
        gold = {d: v["gold"] for d, v in folds[fold].items()}
        pred = {d: spans_from_tags(v["text"], v, v["y"], tags) for d, v in folds[fold].items()}
        sc = strict_scores(gold, pred, labels)
        miss = []
        for d, v in folds[fold].items():
            p = {tuple(x) for x in pred[d]}
            for s, e, l, *_ in v["gold"]:
                if (s, e, l) not in p:
                    miss.append((d, s, e, l, v["text"][max(0, s - 15):e + 15]))
        rep[fold] = dict(upper_bound_macro_f1=sc["macro_f1"], n_gold=sum(len(v["gold"]) for v in folds[fold].values()),
                         n_unrecoverable=len(miss), examples=miss[:12])
    ntok = {f: int(sum(len(v["ids"]) for v in folds[f].values())) for f in folds}
    rep["tokens_per_fold"] = ntok
    return rep


# ============================================================== torch parts
import torch                                   # noqa: E402
import torch.nn as nn                          # noqa: E402
import torch.nn.functional as F                # noqa: E402


def load_phase1(txl_model_py, ckpt_path):
    import importlib.util
    spec = importlib.util.spec_from_file_location("txl_model_p2", txl_model_py)
    mod = importlib.util.module_from_spec(spec)
    sys.modules["txl_model_p2"] = mod
    spec.loader.exec_module(mod)
    ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    cfg = mod.TXLConfig(**ck["config"])
    m = mod.TransformerXL(cfg)
    m.load_state_dict(ck["model"])
    return m, mod


class LowRankLinear(nn.Module):
    """mode='lora'  : y = W x + b + s * B(A x)
       mode='lorma' : y = (I + s * B A) (W x) + b       (LoRMA_+, multiplicative, identity at init)"""

    def __init__(self, base: nn.Linear, r, alpha, dropout, mode):
        super().__init__()
        self.base, self.mode, self.s = base, mode, alpha / r
        for p in base.parameters():
            p.requires_grad_(False)
        din = base.in_features if mode == "lora" else base.out_features
        self.A = nn.Parameter(torch.empty(r, din))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        if self.mode == "lora":
            return self.base(x) + self.s * F.linear(F.linear(self.drop(x), self.A), self.B)
        y0 = F.linear(x, self.base.weight)
        y = y0 + self.s * F.linear(F.linear(self.drop(y0), self.A), self.B)
        return y if self.base.bias is None else y + self.base.bias


class Adapter(nn.Module):
    """Houlsby-style bottleneck adapter applied to the output of a sub-layer projection."""

    def __init__(self, base: nn.Linear, bottleneck):
        super().__init__()
        self.base = base
        for p in base.parameters():
            p.requires_grad_(False)
        d = base.out_features
        self.down = nn.Linear(d, bottleneck)
        self.up = nn.Linear(bottleneck, d)
        nn.init.normal_(self.down.weight, 0, 0.02); nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)

    def forward(self, x):
        y = self.base(x)
        return y + self.up(F.gelu(self.down(y)))


class TXLTagger(nn.Module):
    """Transformer-XL (Phase-1 weights) used as a token encoder.
    attn='bidir' : every token sees left AND right context; relative distances are extended to negative
                   values (the same sinusoid / r_net / v-bias parametrisation, just evaluated at d<0).
    attn='causal': exactly the Phase-1 computation (verified equal in the notebook test)."""

    def __init__(self, base, n_tags, attn="bidir", layers=None, dropout=0.1, head="linear"):
        super().__init__()
        self.attn_mode = attn
        self.emb, self.pos_emb, self.ln_f = base.emb, base.pos_emb, base.ln_f
        keep = list(range(len(base.blocks))) if layers is None else list(layers)
        self.layer_ids = keep
        self.blocks = nn.ModuleList([base.blocks[i] for i in keep])
        self.d = base.cfg.d_model
        for m in self.blocks.modules():
            if isinstance(m, nn.Dropout):
                m.p = dropout
        self.drop = nn.Dropout(dropout)
        self.head_type = head
        if head == "bilstm":
            self.lstm = nn.LSTM(self.d, 256, batch_first=True, bidirectional=True)
            self.head = nn.Linear(512, n_tags)
        else:
            self.head = nn.Linear(self.d, n_tags)

    def _block(self, blk, h, r, gidx, mask):
        a = blk.attn
        B, L, _ = h.shape
        H, dh = a.H, a.dh
        x = blk.ln1(h)
        q = a.q_net(x).view(B, L, H, dh).transpose(1, 2)
        kv = a.kv_net(x).view(B, L, 2, H, dh)
        k, v = kv[:, :, 0].transpose(1, 2), kv[:, :, 1].transpose(1, 2)
        rk = a.r_net(r).view(-1, H, dh).transpose(0, 1).unsqueeze(0)          # [1,H,2L-1,dh]
        qu = (q + a.u[None, :, None, :]) * a.scale
        qv = (q + a.v[None, :, None, :]) * a.scale
        AC = torch.matmul(qu, k.transpose(-1, -2))                              # [B,H,L,L]
        BDf = torch.matmul(qv, rk.transpose(-1, -2))                            # [B,H,L,2L-1]
        BD = BDf.gather(-1, gidx.expand(B, H, L, L))
        score = (AC + BD).float().masked_fill(mask, -1e30)
        prob = a.dropatt(F.softmax(score, dim=-1)).to(v.dtype)
        out = torch.matmul(prob, v).transpose(1, 2).reshape(B, L, H * dh)
        h = h + a.drop(a.o_net(out))
        h = h + blk.drop(blk.ff2(blk.drop(F.gelu(blk.ff1(blk.ln2(h))))))
        return h

    def hidden(self, ids, attn_mask):
        """ids [B,L] long, attn_mask [B,L] bool (True = real token) -> final hidden [B,L,D]."""
        B, L = ids.shape
        dev = ids.device
        h = self.emb(ids)
        pos = torch.arange(L - 1, -L, -1, device=dev, dtype=torch.float32)       # distance d = L-1-c
        r = self.pos_emb(pos)
        i = torch.arange(L, device=dev)
        gidx = ((L - 1) - i[:, None] + i[None, :])[None, None]                   # c for distance i-j
        mask = (~attn_mask)[:, None, None, :]                                    # [B,1,1,L]
        if self.attn_mode == "causal":
            mask = mask | torch.ones(L, L, dtype=torch.bool, device=dev).triu(1)[None, None]
        for blk in self.blocks:
            h = self._block(blk, h, r, gidx, mask)
        return self.ln_f(h)

    def forward(self, ids, attn_mask):
        h = self.hidden(ids, attn_mask)
        if self.head_type == "bilstm":
            h, _ = self.lstm(self.drop(h))
        return self.head(self.drop(h))


def apply_method(model, method, J):
    """Freeze / wrap according to the adaptation method. The head is always trainable."""
    for p in model.parameters():
        p.requires_grad_(method in ("full", "kd"))
    if method == "partial":
        for blk in list(model.blocks)[-J["partial_top_layers"]:]:
            for p in blk.parameters():
                p.requires_grad_(True)
        for p in model.ln_f.parameters():
            p.requires_grad_(True)
    elif method in ("lora", "lorma"):
        tgt = J["lowrank_targets"]
        for blk in model.blocks:
            a = blk.attn
            for name in tgt:
                holder = a if name in ("q_net", "kv_net", "o_net", "r_net") else blk
                setattr(holder, name, LowRankLinear(getattr(holder, name), J["rank"], J["alpha"], J["lowrank_dropout"], method))
    elif method == "adapter":
        for blk in model.blocks:
            blk.attn.o_net = Adapter(blk.attn.o_net, J["adapter_dim"])
            blk.ff2 = Adapter(blk.ff2, J["adapter_dim"])
    elif method == "bitfit":
        for n, p in model.named_parameters():
            if n.endswith("bias") or n.endswith(".u") or n.endswith(".v") or ".ln" in n or n.startswith("ln_f"):
                p.requires_grad_(True)
    for p in model.head.parameters():
        p.requires_grad_(True)
    if getattr(model, "lstm", None) is not None:
        for p in model.lstm.parameters():
            p.requires_grad_(True)
    return model


def count_params(model):
    tot = sum(p.numel() for p in model.parameters())
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return tot, tr


# ============================================================== windows / batching
def windows_for(n, L, stride):
    if n <= L:
        return [0]
    st = list(range(0, n - L + 1, stride))
    if st[-1] + L < n:
        st.append(n - L)
    return st


def make_train_windows(docs, L, stride):
    W = []
    for d in docs.values():
        n = len(d["ids"])
        for s in windows_for(n, L, stride):
            W.append((d["ids"][s:s + L], d["y"][s:s + L]))
    return W


def collate(items, with_y=True):
    L = max(len(x[0]) for x in items)
    ids = torch.zeros(len(items), L, dtype=torch.long)
    am = torch.zeros(len(items), L, dtype=torch.bool)
    y = torch.full((len(items), L), -100, dtype=torch.long)
    for b, it in enumerate(items):
        n = len(it[0])
        ids[b, :n] = torch.as_tensor(it[0], dtype=torch.long)
        am[b, :n] = True
        if with_y:
            y[b, :n] = torch.as_tensor(it[1], dtype=torch.long)
    return ids, am, y


@torch.no_grad()
def doc_logprobs(model, docs, L, stride, bs, device, amp=True):
    """Merge per-window log-probs into one [n, T] array per doc (triangular weights: tokens near a window
    edge get little weight, so every token is mostly judged with context on both sides)."""
    model.eval()
    jobs = []
    for did, d in docs.items():
        n = len(d["ids"])
        if n:
            for s in windows_for(n, L, stride):
                jobs.append((did, s, min(L, n - s)))
    T = model.head.out_features
    acc = {did: np.zeros((len(d["ids"]), T), np.float32) for did, d in docs.items()}
    wsum = {did: np.zeros((len(d["ids"]), 1), np.float32) for did, d in docs.items()}
    jobs.sort(key=lambda j: -j[2])
    for k in range(0, len(jobs), bs):
        chunk = jobs[k:k + bs]
        items = [(docs[did]["ids"][s:s + n], None) for did, s, n in chunk]
        ids, am, _ = collate(items, with_y=False)
        ids, am = ids.to(device), am.to(device)
        with torch.autocast("cuda", dtype=torch.float16, enabled=amp and device.type == "cuda"):
            lg = model(ids, am)
        lp = F.log_softmax(lg.float(), -1).cpu().numpy()
        for b, (did, s, n) in enumerate(chunk):
            pos = np.arange(n)
            w = np.minimum(pos + 1, n - pos).astype(np.float32)[:, None]
            if s == 0:
                w[: n // 2] = np.maximum(w[: n // 2], n / 2)        # doc start: no left context exists anyway
            if s + n == len(docs[did]["ids"]):
                w[n // 2:] = np.maximum(w[n // 2:], n / 2)          # doc end
            acc[did][s:s + n] += lp[b, :n] * w
            wsum[did][s:s + n] += w
    return {did: acc[did] / np.maximum(wsum[did], 1e-6) for did in docs}


def decode_docs(docs, lps, tags, use_viterbi=True):
    allowed, start_ok = bio_transitions(tags)
    pred = {}
    for did, d in docs.items():
        lp = lps[did]
        tg = viterbi(lp, allowed, start_ok) if use_viterbi else lp.argmax(-1)
        pred[did] = spans_from_tags(d["text"], d, tg, tags)
    return pred


# ============================================================== training
def class_weights(windows, T, mode):
    if mode == "none":
        return None
    cnt = np.bincount(np.concatenate([w[1] for w in windows]), minlength=T).astype(np.float64)
    w = np.ones(T)
    ent = cnt[1:]
    mean = ent[ent > 0].mean()
    w[1:] = np.clip(np.sqrt(mean / np.maximum(ent, 1)), 0.5, 4.0)
    return torch.tensor(w, dtype=torch.float32)


def build_model(J, base_factory, n_tags, method):
    base = base_factory()
    layers = J["kd_student_layers"] if method == "kd" else None
    m = TXLTagger(base, n_tags, attn=J["attn"], layers=layers, dropout=J["dropout"], head=J["head"])
    return apply_method(m, method, J)


def train_one(J, model, train_docs, val_docs, tags, labels, device, epochs, log, teacher=None):
    torch.manual_seed(J["seed"]); np.random.seed(J["seed"])
    model.to(device)
    W = make_train_windows(train_docs, J["max_len"], J["train_stride"])
    cw = class_weights(W, len(tags), J["class_weight"])
    cw = cw.to(device) if cw is not None else None
    head_names = {n for n, _ in model.named_parameters() if n.startswith("head") or n.startswith("lstm")}
    new_names = {n for n, _ in model.named_parameters() if any(k in n for k in (".A", ".B", ".down.", ".up."))}
    g_head = [p for n, p in model.named_parameters() if p.requires_grad and (n in head_names or n in new_names)]
    g_body = [p for n, p in model.named_parameters() if p.requires_grad and n not in head_names and n not in new_names]
    groups = [dict(params=g_head, lr=J["head_lr"])]
    if g_body:
        groups.append(dict(params=g_body, lr=J["lr"]))
    opt = torch.optim.AdamW(groups, weight_decay=J["weight_decay"])
    for g in opt.param_groups:
        g["base_lr"] = g["lr"]
    steps_per_epoch = math.ceil(len(W) / J["batch_size"])
    total = steps_per_epoch * epochs
    warm = max(1, int(0.06 * total))
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    best = dict(f1=-1, epoch=0, state=None, report=None, pred=None)
    hist, step, bad = [], 0, 0
    t0 = time.time()
    if use_amp:
        torch.cuda.reset_peak_memory_stats(device)
    for ep in range(1, epochs + 1):
        model.train()
        perm = np.random.permutation(len(W))
        tl, tn = 0.0, 0
        for k in range(0, len(W), J["batch_size"]):
            ids, am, y = collate([W[i] for i in perm[k:k + J["batch_size"]]])
            ids, am, y = ids.to(device), am.to(device), y.to(device)
            f = (step + 1) / warm if step < warm else max(0.0, (total - step) / max(1, total - warm))   # warmup + linear decay
            for g in opt.param_groups:
                g["lr"] = g["base_lr"] * f
            with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                lg = model(ids, am)
            lg = lg.float()
            loss = F.cross_entropy(lg.view(-1, lg.size(-1)), y.view(-1), weight=cw, ignore_index=-100)
            if teacher is not None:
                with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                    tl_ = teacher(ids, am).float()
                Tt = J["kd_temperature"]
                kl = F.kl_div(F.log_softmax(lg / Tt, -1), F.log_softmax(tl_ / Tt, -1), log_target=True, reduction="none").sum(-1)
                kl = (kl * am).sum() / am.sum()
                loss = J["kd_alpha"] * loss + (1 - J["kd_alpha"]) * (Tt ** 2) * kl
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_([p for g in opt.param_groups for p in g["params"]], 1.0)
            scaler.step(opt); scaler.update()
            tl += loss.item(); tn += 1; step += 1
        rec = dict(epoch=ep, train_loss=tl / max(tn, 1), secs=time.time() - t0)
        if val_docs:
            lps = doc_logprobs(model, val_docs, J["max_len"], J["eval_stride"], J["eval_batch_size"], device)
            pred = decode_docs(val_docs, lps, tags)
            sc = strict_scores({d: v["gold"] for d, v in val_docs.items()}, pred, labels)
            rec.update(val_f1=sc["macro_f1"], val_p=sc["macro_p"], val_r=sc["macro_r"])
            if sc["macro_f1"] > best["f1"]:
                best.update(f1=sc["macro_f1"], epoch=ep, report=sc, pred=pred,
                            state={n: p.detach().cpu().clone() for n, p in model.state_dict().items()})
                bad = 0
            else:
                bad += 1
        hist.append(rec)
        log(json.dumps(rec))
        if val_docs and bad >= J["patience"]:
            log(f"early stop at epoch {ep} (best epoch {best['epoch']}, val F1 {best['f1']:.4f})")
            break
    info = dict(train_secs=time.time() - t0, history=hist, n_windows=len(W), steps=step,
                peak_mem_gb=(torch.cuda.max_memory_allocated(device) / 2**30) if use_amp else None)
    return best, info


# ============================================================== one experiment (script entry point)
def run_job(job_path):
    J = json.load(open(job_path))
    out = Path(J["out_dir"]); out.mkdir(parents=True, exist_ok=True)
    logf = open(out / "log.txt", "a")

    def log(msg):
        line = f"[{time.strftime('%H:%M:%S')}][{J['name']}] {msg}"
        print(line, flush=True); logf.write(line + "\n"); logf.flush()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    D = pickle.load(open(J["cache"], "rb"))
    labels, tags, folds = D["labels"], D["tags"], D["folds"]
    if J.get("quick"):
        folds = {f: dict(list(v.items())[: J["quick_docs"]]) for f, v in folds.items()}
    method = J["method"]
    base_factory = lambda: load_phase1(J["txl_model_py"], J["phase1_ckpt"])[0]
    log(f"method={method} attn={J['attn']} head={J['head']} device={device}")

    def get_teacher(which):
        tp = Path(J["kd_teacher_dir"]) / f"{which}_model.pt"
        assert tp.exists(), f"KD needs the teacher checkpoint {tp} (run the teacher job first)"
        tj = json.load(open(Path(J["kd_teacher_dir"]) / "job.json"))
        t = build_model(tj, base_factory, len(tags), tj["method"])
        t.load_state_dict(torch.load(tp, map_location="cpu"))
        t.to(device).eval()
        for p in t.parameters():
            p.requires_grad_(False)
        return t

    results = dict(name=J["name"], method=method, attn=J["attn"], head=J["head"], config=J)
    # ---- (1) model selection: train fold1 -> validate fold2
    model = build_model(J, base_factory, len(tags), method)
    results["params_total"], results["params_trainable"] = count_params(model)
    log(f"params total={results['params_total']/1e6:.2f}M trainable={results['params_trainable']/1e6:.3f}M")
    teacher = get_teacher("val") if method == "kd" else None
    if teacher is not None:
        results["teacher_params"] = count_params(teacher)[0]
    best, info = train_one(J, model, folds["fold1"], folds["fold2"], tags, labels, device, J["epochs"], log, teacher)
    results.update(val_macro_f1=best["f1"], best_epoch=best["epoch"], val_report=best["report"], val_train=info,
                   val_errors=error_breakdown({d: v["gold"] for d, v in folds["fold2"].items()}, best["pred"]))
    json.dump(best["pred"], open(out / "val_pred_fold2.json", "w"))
    log(f"VALIDATION fold1->fold2 strict macro-F1 = {best['f1']:.4f} (epoch {best['epoch']})")
    if J["save_model"]:
        torch.save(best["state"], out / "val_model.pt")
    del teacher
    # ---- (2) final model
    if J["final_fit"] == "all":
        epochs = max(1, best["epoch"] + J.get("final_extra_epochs", 0))
        model = build_model(J, base_factory, len(tags), method)
        teacher = get_teacher("final") if method == "kd" else None
        train_docs = {**folds["fold1"], **folds["fold2"]}
        _, finfo = train_one(J, model, train_docs, None, tags, labels, device, epochs, log, teacher)
        results["final_train"] = finfo
        del teacher
    else:
        model.load_state_dict(best["state"])
    model.to(device)
    if J["save_model"]:
        torch.save({n: p.detach().cpu() for n, p in model.state_dict().items()}, out / "final_model.pt")
    # ---- (3) test prediction
    t0 = time.time()
    test = folds["fold3"]
    lps = doc_logprobs(model, test, J["max_len"], J["eval_stride"], J["eval_batch_size"], device)
    pred = decode_docs(test, lps, tags)
    results["test_infer_secs"] = time.time() - t0
    all_ids = list(D["folds"]["fold3"].keys())
    final = {did: [[int(s), int(e), l] for s, e, l in pred.get(did, [])] for did in all_ids}
    assert set(final) == set(all_ids)
    for did, spans in final.items():
        n = len(D["folds"]["fold3"][did]["text"])
        assert all(0 <= s < e <= n and l in labels for s, e, l in spans), did
    pj = out / "prediction.json"
    json.dump(final, open(pj, "w"))
    zp = out / f"submission_{J['name']}.zip"
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(pj, arcname="prediction.json")
    results["test_entities"] = sum(len(v) for v in final.values())
    results["test_label_counts"] = dict(Counter(l for v in final.values() for _, _, l in v))
    json.dump(results, open(out / "results.json", "w"), indent=1, default=str)
    log(f"DONE -> {zp} | test entities={results['test_entities']} | {json.dumps(results['test_label_counts'])}")


if __name__ == "__main__":
    run_job(sys.argv[1])


Writing /kaggle/working/p2/p2_ner.py


In [3]:
# ============================================================ 2. PREP + ALIGNMENT CHECK + MODEL CHECK
sys.path.insert(0, str(WORK))
import importlib, p2_ner as P; importlib.reload(P)
import torch, numpy as np

tok = P.load_bpe(TOK_PY, TOK_PATH)
print("tokenizer kind:", tok.kind, "| vocab", tok.vocab_size); assert tok.vocab_size == 32000
CACHE = WORK / "cache.pkl"
t0 = time.time()
folds = P.prepare(DATA, tok, LABELS, CACHE)
print(f"tokenised all folds in {time.time()-t0:.0f}s")

rep = P.alignment_report(folds, LABELS)
for f in ["fold1", "fold2"]:
    r = rep[f]
    print(f"{f}: alignment upper-bound strict macro-F1 = {r['upper_bound_macro_f1']:.4f} | gold {r['n_gold']} | unrecoverable {r['n_unrecoverable']}")
    for ex in r["examples"][:6]: print("    ", ex)
print("tokens per fold:", rep["tokens_per_fold"])
assert rep["fold1"]["upper_bound_macro_f1"] > 0.97, "alignment is losing too many spans - send me this output"
# token ids must be exactly what Phase-1 pretraining produced (ASCII-normalised text -> my tokenizer)
for d in list(folds["fold1"].values())[:5]:
    norm = P.normalize_with_map(d["text"])[0]
    ref = tok.tok.encode(norm) if tok.kind == "new" else tok.encode_full(norm)
    assert list(d["ids"]) == list(ref), "token ids differ from the Phase-1 tokenizer"
print("token ids identical to Phase-1 tokenisation: OK")

# encoder in causal mode == Phase-1 Transformer-XL forward (on a real document window)
base, _ = P.load_phase1(TXL_MODEL_PY, PHASE1_CKPT); base.eval()
print(f"Phase-1 model: {base.num_params()/1e6:.2f}M params | config {base.cfg}")
d0 = next(iter(folds["fold1"].values()))
ids = torch.as_tensor(d0["ids"][:256], dtype=torch.long)[None]
tg = P.TXLTagger(base, 25, attn="causal", dropout=0.0).eval()
with torch.no_grad():
    ref = base(ids)["logits"]
    mine = torch.nn.functional.linear(tg.hidden(ids, torch.ones_like(ids, dtype=torch.bool)), base.emb.weight, base.out_bias)
diff = (ref - mine).abs().max().item(); print("causal encoder vs Phase-1 forward, max |diff| =", diff); assert diff < 1e-3
del base, tg


tokenizer kind: new | vocab 32000
tokenised all folds in 3s
fold1: alignment upper-bound strict macro-F1 = 0.9887 | gold 4120 | unrecoverable 69
     ('115651329', 10266, 10299, 'PREC', 'In the case of State of M.P. vs. Kashiram & Ors.[1], the scope ')
     ('975074', 7995, 8085, 'PREC', '6 SC 1565] and Gambangi Appalaswamy Naldu and Others v. Behara Venkataramanayya Patro [AIR 1984- SC 1728]. Even otherwis')
     ('736324', 1963, 1976, 'AUTH', 'truck down the Railway Boards instructions ')
     ('736324', 11218, 11231, 'AUTH', 'lementation of Railway Boards decision to e')
     ('736324', 12849, 12901, 'PREC', 'rlier cases in Shankarsan Dash Vs. Union of India [1991 (3) SCC 47]. Para 7 of the')
     ('736324', 14121, 14174, 'PREC', ' 4 SCC 268] or Jatendra Kumar vs. State of Punjab [(1985) 1 SCC 122]. Hence the con')
fold2: alignment upper-bound strict macro-F1 = 0.9848 | gold 4273 | unrecoverable 85
     ('76640285', 18, 41, 'COURT', 'ORTABLE IN THE SUPREME COURT OF INDIA CRIMINAL AP

In [4]:
# ============================================================ 3. EXPERIMENTS
COMMON = dict(
    cache=str(CACHE), txl_model_py=TXL_MODEL_PY, phase1_ckpt=PHASE1_CKPT,
    attn="bidir",              # 'bidir' (sees both sides; recommended) | 'causal' (= Phase-1 attention, ablation)
    head="linear",             # 'linear' | 'bilstm'
    max_len=512, train_stride=256, eval_stride=256, batch_size=8, eval_batch_size=16,
    lr=1.5e-4, head_lr=1e-3, weight_decay=0.01, dropout=0.1,
    epochs=15, patience=5, class_weight="sqrt", seed=781,
    lowrank_targets=["q_net", "kv_net", "o_net", "ff1", "ff2"], rank=16, alpha=32, lowrank_dropout=0.05,
    adapter_dim=64, partial_top_layers=4,
    kd_student_layers=[0, 3, 6, 9, 11, 13], kd_temperature=2.0, kd_alpha=0.5,
    final_fit="all",           # 'all': retrain on fold1+fold2 for best-epoch epochs | 'best': use the fold1 model
    save_model=True, quick=QUICK_TEST, quick_docs=4,
)
#            name        method     overrides
JOBS = {
  "full":    ("full",    dict()),                                              # also the KD teacher
  "lorma":   ("lorma",   dict(epochs=25, head_lr=1e-3, save_model=False)),     # compulsory
  "lora":    ("lora",    dict(epochs=25, head_lr=1e-3, save_model=False)),
  "kd":      ("kd",      dict(lr=2e-4, kd_teacher_dir=str(OUT / "full"), save_model=False)),
  "adapter": ("adapter", dict(epochs=25, head_lr=1e-3, save_model=False)),
  "bitfit":  ("bitfit",  dict(epochs=25, lr=2e-3, head_lr=1e-3, save_model=False)),
  # extra ablations (uncomment if time allows):
  # "lorma_causal": ("lorma", dict(attn="causal", epochs=25, save_model=False)),
  # "full_bilstm":  ("full",  dict(head="bilstm", save_model=False)),
  # "partial":      ("partial", dict(lr=2e-4, save_model=False)),
}
# two GPUs -> two queues (kd must run after full on the same queue)
QUEUES = [["full", "kd", "bitfit"], ["lorma", "lora", "adapter"]]
if QUICK_TEST:
    COMMON.update(epochs=2, patience=2)
    QUEUES = [["full", "kd"], ["lorma"]]

n_gpu = torch.cuda.device_count(); print("GPUs:", n_gpu)
if n_gpu < 2: QUEUES = [sum(QUEUES, [])]

def write_job(name):
    meth, over = JOBS[name]
    J = dict(COMMON, name=name, method=meth, out_dir=str(OUT / name), **over)
    if QUICK_TEST: J.update(epochs=2, patience=2)
    Path(J["out_dir"]).mkdir(parents=True, exist_ok=True)
    p = Path(J["out_dir"]) / "job.json"; json.dump(J, open(p, "w"), indent=1)
    return p

def run_queue(names, gpu):
    for n in names:
        if (OUT / n / "results.json").exists():
            print(f"[skip] {n} already done"); continue
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED="1")
        p = subprocess.Popen([sys.executable, str(WORK / "p2_ner.py"), str(write_job(n))], env=env,
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            if "Warning" not in line: print(f"[gpu{gpu}] {line}", end="")
        p.wait()
        if p.returncode != 0:
            print(f"[gpu{gpu}] !!! job {n} FAILED (see {OUT/n/'log.txt'} and the lines above)")

T0 = time.time()
threads = [threading.Thread(target=run_queue, args=(q, g)) for g, q in enumerate(QUEUES)]
for t in threads: t.start()
for t in threads: t.join()
print(f"all jobs finished in {(time.time()-T0)/60:.1f} min")


GPUs: 2
[gpu1] [21:40:37][lorma] method=lorma attn=bidir head=linear device=cuda
[gpu0] [21:40:37][full] method=full attn=bidir head=linear device=cuda
[gpu1] [21:40:38][lorma] params total=66.25M trainable=2.077M
[gpu0] [21:40:38][full] params total=64.19M trainable=64.187M
[gpu0] [21:41:38][full] {"epoch": 1, "train_loss": 0.599416252530434, "secs": 38.278528451919556, "val_f1": 0.3862977566126045, "val_p": 0.37443068754121134, "val_r": 0.4239634118224844}
[gpu1] [21:41:40][lorma] {"epoch": 1, "train_loss": 0.9580488889732144, "secs": 38.42395782470703, "val_f1": 0.2969784073412198, "val_p": 0.3377447581105744, "val_r": 0.32548801848657816}
[gpu0] [21:42:38][full] {"epoch": 2, "train_loss": 0.06313803532076152, "secs": 95.63295388221741, "val_f1": 0.4527079043432826, "val_p": 0.46222611068425484, "val_r": 0.502821297241168}
[gpu1] [21:42:40][lorma] {"epoch": 2, "train_loss": 0.11856608969921416, "secs": 96.78159141540527, "val_f1": 0.3753103119864986, "val_p": 0.44045222783192023, "v

In [5]:
# ============================================================ 4. SUMMARY + SUBMISSIONS
import pandas as pd, zipfile
rows, per = [], {}
SUB = Path("/kaggle/working/submissions" + ("_quick" if QUICK_TEST else "")); SUB.mkdir(exist_ok=True)
test_ids = set(json.load(open(DATA / "Test" / "fold3-text.json")).keys())
for n in JOBS:
    rp = OUT / n / "results.json"
    if not rp.exists(): continue
    r = json.load(open(rp))
    z = OUT / n / f"submission_{n}.zip"
    with zipfile.ZipFile(z) as zz:
        assert zz.namelist() == ["prediction.json"]
        pj = json.loads(zz.read("prediction.json")); assert set(pj) == test_ids
    shutil.copy(z, SUB / z.name)
    rows.append(dict(exp=n, method=r["method"], attn=r["attn"], head=r["head"],
                     val_macroF1=round(r["val_macro_f1"], 4), val_P=round(r["val_report"]["macro_p"], 4), val_R=round(r["val_report"]["macro_r"], 4),
                     best_epoch=r["best_epoch"], params_M=round(r["params_total"]/1e6, 2), trainable_M=round(r["params_trainable"]/1e6, 3),
                     peak_mem_GB=round(r["val_train"]["peak_mem_gb"] or 0, 2), train_min=round(r["val_train"]["train_secs"]/60, 1),
                     final_train_min=round(r.get("final_train", {}).get("train_secs", 0)/60, 1), test_infer_s=round(r["test_infer_secs"], 1),
                     test_entities=r["test_entities"], fp_types=r["val_errors"]["fp_kinds"]))
    per[n] = {l: round(v["f1"], 3) for l, v in r["val_report"]["per_label"].items()}
df = pd.DataFrame(rows).sort_values("val_macroF1", ascending=False)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
display(df)
display(pd.DataFrame(per).T)            # per-label F1 on fold2, per experiment
df.to_csv(WORK / ("summary_quick.csv" if QUICK_TEST else "summary.csv"), index=False)
print("submission zips:", sorted(p.name for p in SUB.glob("*.zip")), "->", SUB)
if QUICK_TEST: print("\nQUICK TEST ONLY - do NOT submit these. Set QUICK_TEST=False and Save & Run All (Commit).")


,exp,method,attn,head,val_macroF1,val_P,val_R,best_epoch,params_M,trainable_M,peak_mem_GB,train_min,final_train_min,test_infer_s,test_entities,fp_types
1,lorma,lorma,bidir,linear,0.5447,0.5472,0.5549,24,66.25,2.077,4.83,25.0,33.0,21.1,4121,"{'boundary_error': 479, 'overlap_other_label':..."
2,lora,lora,bidir,linear,0.5337,0.5162,0.5633,14,66.14,1.963,4.50,18.1,18.0,19.7,4314,"{'spurious': 940, 'boundary_error': 503, 'labe..."
0,full,full,bidir,linear,0.5225,0.5150,0.5438,9,64.19,64.187,5.03,13.5,12.4,18.2,4261,"{'label_error': 332, 'boundary_error': 525, 's..."
4,adapter,adapter,bidir,linear,0.4949,0.4948,0.5246,7,66.04,1.864,3.78,10.2,7.8,18.4,4575,"{'boundary_error': 677, 'overlap_other_label':..."
3,kd,kd,bidir,linear,0.4859,0.5178,0.4799,13,36.88,36.879,2.81,10.2,14.1,9.2,3922,"{'boundary_error': 522, 'overlap_other_label':..."
5,bitfit,bitfit,bidir,linear,0.4703,0.4504,0.5077,18,64.19,0.093,3.82,19.4,20.1,18.1,4845,"{'boundary_error': 679, 'overlap_other_label':..."


,APP,RESP,A.COUNSEL,R.COUNSEL,JUDGE,WIT,AUTH,COURT,STAT,PREC,DATE,CASENO
full,0.216,0.189,0.471,0.511,0.664,0.374,0.357,0.815,0.721,0.451,0.952,0.549
lorma,0.302,0.221,0.568,0.511,0.697,0.404,0.316,0.811,0.748,0.437,0.950,0.572
lora,0.251,0.215,0.574,0.479,0.727,0.270,0.406,0.808,0.753,0.476,0.941,0.505
kd,0.235,0.183,0.357,0.393,0.707,0.273,0.346,0.836,0.714,0.371,0.926,0.489
adapter,0.192,0.280,0.508,0.437,0.590,0.291,0.373,0.793,0.692,0.399,0.950,0.433
bitfit,0.236,0.185,0.364,0.426,0.552,0.295,0.393,0.826,0.658,0.404,0.940,0.364


submission zips: ['submission_adapter.zip', 'submission_bitfit.zip', 'submission_full.zip', 'submission_kd.zip', 'submission_lora.zip', 'submission_lorma.zip'] -> /kaggle/working/submissions
